## MonReader - part 5

----

### Generative Text-to-Speech with Sesame (CSM)

**Objective.**  
Explore and integrate **Sesame (Conversational Speech Model – CSM)** as a **generative Text-to-Speech engine** for long-form literary prose, building on the **canonical, sentence-aware chunks** produced in **MonReader – Part 4**.

We continue using the same two books:
- *The Chamber* — John Grisham *(English)*
- *A onda que se ergueu no mar* — Ruy Castro *(Portuguese)*

This notebook is intentionally **hands-on and exploratory**: we focus on understanding Sesame’s **input expectations**, **GPU inference workflow** (PyTorch / Hugging Face), and **stability on narrative text**, rather than benchmarking multiple TTS systems.

### Why Sesame?

Sesame represents a newer class of **generative speech models**, aiming for natural pacing and expressive prosody by modeling speech more holistically than classic TTS pipelines.

We explore Sesame because it promises:
- **Natural prosody and rhythm**
- **Better long-form behavior** than many traditional TTS stacks
- A research-friendly setup for inspection and experimentation

### What We Do in This Part

- Set up the environment and verify **GPU execution**
- Run minimal synthesis to learn the **API + audio outputs**
- Test controlled samples from both books (English vs. Portuguese, short vs. long chunks)
- Record observations and define a clean **integration path** back into MonReader

> The goal here is correctness and understanding: learn how to use Sesame reliably for long-form audiobook-style synthesis, and identify its practical limits early.


---

## Step J.1 — Environment Setup + GPU Verification

This step ensures:
- packages are available (Transformers w/ CSM support)
- GPU is visible to PyTorch (CUDA)
- we capture a reproducible environment report

In [1]:
from pathlib import Path
import os
import sys
import platform
import subprocess
import json
import time

In [3]:
BASE = Path.cwd()
WORK_DIR = BASE / "work"

# Part 5 outputs
STEP5_DIR = WORK_DIR / "step5_sesame_csm"
STEP5_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
# Verify PyTorch and CUDA version

import torch, transformers
from transformers import AutoProcessor, CsmForConditionalGeneration
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)


Torch: 2.5.1+cu121
CUDA available: True
Transformers: 4.57.3


Run `huggingface-cli login` on the terminal:

```

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|
``` 

```
A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
Setting a new token will erase the existing one.
To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Token can be pasted using 'Right-Click'.

Enter your token (input will not be visible):
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
```


### Load Sesame model + processor

This will download weights on first run (~1B params)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [13]:
MODEL_ID = "sesame/csm-1b"

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model...")
model = CsmForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()

print("Model loaded.")


Loading processor...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.19it/s]


Model loaded.


### Sesame (CSM) — Model Overview and Architecture

#### What is Sesame (CSM)?

**Sesame (Conversational Speech Model, CSM)** is a **fully generative text-to-speech model** that produces natural-sounding speech by modeling audio as a sequence of discrete tokens, rather than relying on a traditional multi-stage TTS pipeline.

Unlike classical TTS systems (text normalization → phonemes → acoustic model → vocoder), Sesame operates as a **single, end-to-end generative model**, conceptually similar to modern large language models, but trained to emit **audio tokens instead of text tokens**.

In the MonReader project, Sesame is used as the **final synthesis stage**, converting clean, sentence-aware OCR text into audiobook-style speech.

---

### The Processor

Example initialization:

    processor = AutoProcessor.from_pretrained("sesame/csm-1b")

The **processor** is a Hugging Face abstraction that encapsulates **all non-learned input and output transformations** required by the model.

For Sesame, the processor fulfills two essential roles:

#### 1. Text preprocessing (input side)

- Accepts raw Unicode text directly (no phoneme conversion required)
- Tokenizes text into a sequence of **text tokens** compatible with the model’s embedding space
- Applies internal normalization consistent with the model’s training configuration

This design allows Sesame to operate directly on canonical prose without language-specific text normalization pipelines.

#### 2. Audio decoding (output side)

- Converts generated **discrete audio tokens** into a continuous waveform
- Uses an internal **neural audio codec (Mimi)** for reconstruction
- Outputs PCM audio as a NumPy array at the correct sampling rate

The processor therefore acts as the **bridge between symbolic text and audible waveform data**.

---

### The Model

Example initialization:

    model = CsmForConditionalGeneration.from_pretrained("sesame/csm-1b")

The **model** is a large-scale neural network (≈1 billion parameters) trained for **conditional sequence generation**, where:

- **Input**: a sequence of text tokens
- **Output**: a sequence of discrete audio tokens

The model itself does not emit waveforms; instead, it predicts audio tokens that are later decoded by the processor.

---

### High-Level Architecture

At a high level, Sesame follows a **Transformer-based encoder–decoder architecture**, adapted for speech generation.

#### Conceptual layout

    Text
     │
     ▼
    Text Tokenizer
     │
     ▼
    Text Embeddings
     │
     ▼
    Transformer Encoder
     │
     ▼
    Cross-Attention
     │
     ▼
    Transformer Decoder
     │
     ▼
    Discrete Audio Tokens
     │
     ▼
    Neural Audio Codec (Mimi)
     │
     ▼
    Waveform (PCM audio)

---

### Key Technical Components

#### Transformer backbone

- Multi-layer Transformer architecture
- Self-attention for long-range dependency modeling
- Cross-attention between text and audio token streams
- Supports coherent prosody across long sentences and paragraphs

#### Discrete audio token modeling

- Speech is represented as sequences of **discrete latent codes**
- These codes correspond to compressed representations learned by the Mimi codec
- This reduces generation complexity compared to raw waveform prediction

#### Neural audio codec (Mimi)

- Learns a compact latent space for speech audio
- Enables high perceptual quality and stable reconstruction
- Decoding is performed after token generation, not during Transformer inference

#### Autoregressive generation

- Audio tokens are generated sequentially
- Output duration is controlled via `max_new_tokens`
- Prosody, pauses, and rhythm emerge from learned patterns rather than explicit rules

---

### Why Sesame Fits MonReader

Sesame aligns well with MonReader’s design goals:

- Works directly on **raw, canonical prose**
- Exhibits **long-form stability** suitable for audiobook-style narration
- Produces **natural prosody and pacing**
- Avoids complex, language-specific TTS pipelines

This supports MonReader’s guiding principle:

> Minimal intervention, maximum narrative fidelity.

---

### Inference Characteristics

- GPU-accelerated (FP16 recommended)
- First inference downloads model shards (~1B parameters)
- Generation cost scales primarily with audio length
- Memory usage dominated by Transformer layers rather than codec decoding

---

